In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms
from tqdm import tqdm
import os 
import pandas as pd
import matplotlib.pyplot as plt
import timm
from PIL import Image
from sklearn.model_selection import train_test_split


In [ ]:
df = pd.read_csv('/kaggle/input/test-emotion-dataset/emotion-recognition-dataset2/data.csv')
image_directory = '/kaggle/input/test-emotion-dataset/emotion-recognition-dataset2/dataset/'

df = df[df['label'] != 'Ahegao']

df['path']= image_directory + df['path'].str.split('/').str[-1]
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

def check_images_exist(df):
    df_filtered = df[df['path'].apply(lambda x: os.path.isfile(x))]
    return df_filtered


df = check_images_exist(df)
df['label'] = df['label'].astype('category')
df['label'] = (df['label'].cat.codes).astype(str)

In [ ]:
test_transform= transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform =  transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.loc[idx, 'path']
        label = int(self.df.loc[idx, 'label'])
        image = Image.open(image_path).convert('RGB')
        
        if image:
            image = self.transform(image)
        return image, label
        
    

In [ ]:
#torch.manual_seed(42)
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42)

In [ ]:
train_dataset = EmotionDataset(train_data, transform)
val_dataset = EmotionDataset(val_data, test_transform)
test_dataset = EmotionDataset(test_data, test_transform)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
def show_images(data_loader):
    images, labels = next(iter(data_loader))
    fig, axes = plt.subplots(1,5, figsize=(15,4))
    for i in range(5):
        img = images[i].permute(1,2, 0).cpu().numpy()
        axes[i].imshow(img)
        axes[i].set_title(f"Клас: {labels[i].item()}")
        axes[i].axis('off')

    plt.show()

show_images(train_loader)
        
        
        

In [ ]:
from timm.data import Mixup

cutmix_or_mixup = Mixup(
    mixup_alpha=0.6,
    cutmix_alpha=0.6,
    cutmix_minmax=None,
    prob=0.5,
    switch_prob=0.5,
    mode='batch',
    label_smoothing=0.2,
    num_classes=5
)

In [ ]:
def train(model, train_loader, val_loader, criterion, val_criterion, optimizer,scheduler, epochs=30, device='cpu'):
    best_model_path = "/kaggle/working/coatnet.pth"
    epoch_without_improvement = 0
    best_val_loss = float('inf')

    train_loss = []
    train_acc = []
    val_loss = []
    val_accuracies = []


    for epoch in range(1, epochs+1):
        total = 0
        model.train()
        total_train_loss,total_train_acc = 0.0, 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            images, labels = cutmix_or_mixup(images, labels)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss +=loss.item()
            _, predicted = outputs.max(1)
            _, targets =labels.max(1)
            total += labels.size(0)
            total_train_acc += predicted.eq(targets).sum().item()

        avg_train_loss = total_train_loss/len(train_loader)
        avg_train_acc = total_train_acc/total

        train_loss.append(avg_train_loss)
        train_acc.append(avg_train_acc)

        model.eval()
        total_val_loss =0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = val_criterion(outputs, labels)
                total_val_loss += loss.item() * images.size(0)  # акумулюємо по семплах
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = total_val_loss / total
        scheduler.step(avg_val_loss)
        accuracy = correct / total
        val_accuracies.append(accuracy)
        val_loss.append(avg_val_loss)
        print(
                f"[epoch {epoch}/{epochs}] "
                f"Train accuracy {avg_train_acc:.4f} | "
                f"Train loss {avg_train_loss:.4f} | "
                f"Val loss {avg_val_loss:.4f} |"
                f"Val accuracy {accuracy:.4f}"
        )


        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f'Best model with updated with loss: {avg_val_loss:.4f}')
            epoch_without_improvement = 0
        else:
            epoch_without_improvement+=1

        if epoch_without_improvement > 3:
            print(f"Early stopping after {epoch} epochs without improvement.")
            return train_loss, train_acc, val_loss, val_accuracies

    return train_loss, train_acc, val_loss, val_accuracies




In [ ]:
class EmotionModel(nn.Module):
    def __init__(self, num_classes=5):
        super(EmotionModel, self).__init__()
        self.backbone = timm.create_model('convnextv2_tiny', pretrained=True, num_classes=0)
        num_feature = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(num_feature, 128),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(128, 5)
        )
    def forward(self, x):
        features = self.backbone(x)
        output = self.classifier(features)
        return output

In [ ]:
from timm.loss import SoftTargetCrossEntropy
model = EmotionModel(num_classes=5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

for param in model.backbone.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True
    
criterion = SoftTargetCrossEntropy()
val_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, min_lr = 1e-8, verbose=True,
)

train_loss, train_acc, val_loss, val_acc = train(model, train_loader, val_loader, criterion, val_criterion,optimizer, scheduler, epochs=30, device=device)

In [ ]:
for param in model.backbone.parameters():
    param.requires_grad =True
for param in model.classifier.parameters():
    param.requires_grad =True

In [ ]:
for i, param in enumerate(model.backbone.parameters()):
    print(i, param.requires_grad)

In [ ]:
from timm.loss import SoftTargetCrossEntropy
criterion = SoftTargetCrossEntropy()
val_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr':0.00001},
    {'params': model.classifier.parameters(),'lr':0.00005}
], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-8, verbose=True
)
train_loss, train_acc, val_loss, val_accuracies = train(
    model, train_loader, val_loader, criterion, val_criterion, optimizer, 
     scheduler, epochs=35, device=device
)

In [ ]:
epochs = range(1, len(train_loss)+1)
plt.figure(figsize=(10,5))
plt.plot(epochs, train_loss, label='train_loss')
plt.plot(epochs, val_loss, label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
epochs = range(1, len(train_loss)+1)
plt.figure(figsize=(10,5))
plt.plot(epochs, train_acc, label='train_acc')
plt.plot(epochs, val_accuracies, label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model = EmotionModel(5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
path= '/kaggle/input/conv-head-emotion/convnext_full_emotion.pth'
model.load_state_dict(torch.load(path, map_location=device))
model = model.to(device)


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix,classification_report
emotion_labels = ['Angry', 'Happy', 'Neutral', 'Sad', 'Surprise']
val_criterion = nn.CrossEntropyLoss()
test_loss= 0.0
correct = 0
total = 0
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = val_criterion(outputs, labels)
        test_loss+=loss.item() * images.size(0)
        _, predicted = torch.max(outputs,1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total +=labels.size(0)
        correct += (predicted == labels).sum().item()
avg_test_acc = correct/total
avg_test_loss = test_loss/total
cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=emotion_labels)

print(f"Test loss: {avg_test_loss:.4f}")
print(f"Test accuracy: {avg_test_acc:.4f}")

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotion_labels, yticklabels=emotion_labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

print(report)

# Results

### Full models, loss:
**ConvNeXt - 0.471**

**Swin     - 0.491**

**CoAtNet  - 0.543**


### Full models, accuracy:
**ConvNeXt - 0.848**

**Swin     - 0.836**

**CoAtNet  - 0.821**
  

### Full models, F1 value:
**ConvNeXt - 0.85**

**Swin     - 0.84**

**CoAtNet  - 0.83**